# Fase 2: Geração do Relatório Metodológico Atualizado e Auditoria Final

Este notebook cumpre duas funções essenciais para a defesa da Iniciação Científica:
1. **Documentação:** Gera o `Relatorio_Metodologico_Fase2.xlsx` contendo o novo De-Para de Variáveis, o Mapa de Arquivos atualizado e o quadro de Limitações e Soluções (Pobreza Multidimensional).
2. **Auditoria de Qualidade:** Lê a base final do IVS (`Base_IVS_Multidimensional_Formatada.xlsx`) e verifica a integridade de linhas, limites matemáticos (0 a 1) e ausência de dados nulos.

In [2]:
import pandas as pd
import os

print("1. Criando os DataFrames para o Relatório Metodológico da Fase 2...")

caminho_bd = '../../banco_de_dados/'

# 1. Mapa de Arquivos (Atualizado com Demografia e Parentesco)
mapa_arquivos = [
    {'Arquivo do Censo 2022': 'Agregados_por_setores_basico_BR_20250417.csv', 'Dimensão do IVS': 'Filtros e População Base', 'Descrição Resumida': 'Espinha dorsal do banco para identificar os setores, áreas rurais/urbanas e população total.'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv', 'Dimensão do IVS': 'Denominador Habitacional e Sem-Abrigo', 'Descrição Resumida': 'Fornece a contagem de domicílios permanentes, improvisados (tendas) e a morfologia da moradia (casa/apartamento).'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv', 'Dimensão do IVS': 'Saneamento Básico e Patrimônio', 'Descrição Resumida': 'Numeradores de infraestrutura (água, esgoto, lixo) e marcador de precariedade sanitária (banheiros).'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_alfabetizacao_BR.csv', 'Dimensão do IVS': 'Educação / Escolaridade', 'Descrição Resumida': 'Contagem de analfabetos com 15 anos ou mais e a população base educacional.'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_cor_ou_raca_BR.csv', 'Dimensão do IVS': 'Vulnerabilidade Social', 'Descrição Resumida': 'Contagem de raças (preta, parda e indígena) para a dimensão demográfica.'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_renda_responsavel_BR.csv', 'Dimensão do IVS': 'Renda (Base Financeira)', 'Descrição Resumida': 'Rendimento nominal médio mensal utilizado como linha base invertida.'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_demografia_BR.csv', 'Dimensão do IVS': 'Sobrecarga Infantil (NOVO)', 'Descrição Resumida': 'Contagem de crianças de 0 a 14 anos para criar a razão de dependência familiar.'},
    {'Arquivo do Censo 2022': 'Agregados_por_setores_parentesco_BR.csv', 'Dimensão do IVS': 'Total de Lares Reais (NOVO)', 'Descrição Resumida': 'Contagem do número de responsáveis. Substitui o número de casas físicas para contornar a coabitação oculta.'}

]
df_mapa = pd.DataFrame(mapa_arquivos)

# 2. De_Para Variáveis (O que mudou em relação a 2012)
de_para = [
    {'Componente IVS': 'Saneamento: Água, Esgoto e Lixo', 'O que o IVS 2012 pedia': 'Variáveis do Censo 2010 (V013, V019, V037...)', 'Variáveis Censo 2022 Equivalentes': 'Água: V00112 a V00118 | Esgoto: V00312 a V00316 | Lixo: V00398 a V00402', 'Observação Metodológica': 'O somatório agrupa fontes inadequadas. Denominador alterado de domicílios totais para Número de Responsáveis (V01042).'},
    {'Componente IVS': 'Habitação: Razão de Moradores', 'O que o IVS 2012 pedia': 'Média de moradores pronta ou divisão direta', 'Variáveis Censo 2022 Equivalentes': 'Numerador: (V00005 + V00006) | Denominador: V01042', 'Observação Metodológica': 'Cálculo manual exigido. Somamos a população de casas e abrigos improvisados e dividimos pelo total de Responsáveis, medindo a real superlotação.'},
    {'Componente IVS': 'Extrema Pobreza Multidimensional', 'O que o IVS 2012 pedia': 'Percentual de famílias até 2 salários mínimos', 'Variáveis Censo 2022 Equivalentes': 'Renda (V06004), Sem Banheiro (V00236, V00238), Improvisados (V00002), Dependentes (V01031 a V01033)', 'Observação Metodológica': 'Substituição metodológica total baseada em Alkire-Foster e IPEA. A Renda Invertida (40%) é corrigida pelos déficits estruturais e demográficos (60%).'},
    {'Componente IVS': 'Educação e Vulnerabilidade Social', 'O que o IVS 2012 pedia': 'Taxa de Analfabetismo e Raça', 'Variáveis Censo 2022 Equivalentes': 'Analfabetismo: V00901 / V00900 | Raça: (V01318 + V01320 + V01321) / v0001', 'Observação Metodológica': 'A proporção mantém-se exata, utilizando o total da população do setor (v0001) para raça e maiores de 15 anos para educação.'}

]
df_depara = pd.DataFrame(de_para)

# 3. Limitações e Soluções Acadêmicas (O Escudo de Defesa)
limitacoes = [
    {'Limitação no Censo 2022 Agregado': 'O IBGE não disponibilizou a contagem absoluta de domicílios dividida por faixas de salário mínimo nos agregados preliminares.', 'Impacto no Projeto': 'Alto (Impossibilidade de aplicar a fórmula exata do IVS de 2012)', 'Solução Científica Aplicada': 'Implementação de um Proxy de Extrema Pobreza Multidimensional. Utilizamos o Rendimento Médio Invertido combinado com marcadores físicos de miséria (ausência de banheiros exclusivos e presença de domicílios improvisados) e demográficos (sobrecarga de 0 a 14 anos), neutralizando a distorção da média salarial.', 'Referência Acadêmica': 'Método Alkire-Foster (OPHI, Oxford); Critério Brasil (ABEP); Notas Técnicas do IPEA 2024/2025.'},
    {'Limitação no Censo 2022 Agregado': 'A variável "População em Domicílios Coletivos" agrupa indiscriminadamente penitenciárias, asilos e quartéis, distorcendo a densidade local.', 'Impacto no Projeto': 'Médio', 'Solução Científica Aplicada': 'Recálculo da densidade habitacional manual usando a variável V01042 (Pessoa Responsável) como universo denominador, contornando a coabitação oculta e excluindo os 100% coletivos.', 'Referência Acadêmica': 'Estudos de Déficit Habitacional Qualitativo (Marques, 2016).'}

]
df_limitacoes = pd.DataFrame(limitacoes)

# Exportação Premium do Relatório Metodológico
caminho_relatorio = caminho_bd + 'Relatorio_Metodologico_Fase2_Atualizado.xlsx'
with pd.ExcelWriter(caminho_relatorio, engine='xlsxwriter') as writer:
    df_mapa.to_excel(writer, sheet_name='Mapa_de_Arquivos', index=False)
    df_depara.to_excel(writer, sheet_name='De_Para_Variaveis', index=False)
    df_limitacoes.to_excel(writer, sheet_name='Limitacoes_e_Solucoes', index=False)
    
    workbook = writer.book
    formato_cabecalho = workbook.add_format({'bold': True, 'bg_color': '#4F81BD', 'font_color': 'white', 'border': 1, 'text_wrap': True})
    formato_texto = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1})
    
    for aba in ['Mapa_de_Arquivos', 'De_Para_Variaveis', 'Limitacoes_e_Solucoes']:
        ws = writer.sheets[aba]
        for col_num, col_name in enumerate(df_mapa.columns if aba == 'Mapa_de_Arquivos' else (df_depara.columns if aba == 'De_Para_Variaveis' else df_limitacoes.columns)):
            ws.write(0, col_num, col_name, formato_cabecalho)
        ws.set_column('A:D', 40, formato_texto)

print(f"Relatório Metodológico da Fase 2 gerado em:\n{caminho_relatorio}")



1. Criando os DataFrames para o Relatório Metodológico da Fase 2...
Relatório Metodológico da Fase 2 gerado em:
../../banco_de_dados/Relatorio_Metodologico_Fase2_Atualizado.xlsx


In [ ]:
# Imprime separadores visuais para destacar o início da auditoria final
print("\n=========================================")
print("      AUDITORIA FINAL DA BASE EXCEL      ")
print("=========================================")

# Lê a base formatada gerada no passo anterior (aba 'Base_Analitica' do Excel)
caminho_excel_ivs = caminho_bd + 'Base_IVS_Multidimensional_Formatada.xlsx'
df_teste = pd.read_excel(caminho_excel_ivs, sheet_name='Base_Analitica')

# Inicializa o contador de erros encontrados na auditoria
erros = 0

# 1. Teste de Volume: verifica se o número de linhas está correto
linhas = len(df_teste)
if linhas == 450088:
    # Se o número de linhas está correto, imprime mensagem de sucesso
    print(f"✅ Volume Perfeito: A base contém exatamente os {linhas} setores válidos previstos.")
else:
    # Caso contrário, alerta sobre o volume incorreto e incrementa o contador de erros
    print(f"❌ Alerta de Volume: A base tem {linhas} linhas. Deveria ter 450088.")
    erros += 1

# 2. Teste da Chave Primária: verifica se há setores sem código (nulos)
nulos_setor = df_teste['CD_SETOR'].isnull().sum()
if nulos_setor == 0:
    # Se não há nulos, integridade da chave está garantida
    print("✅ Integridade Chave: Nenhum setor censitário perdeu o seu ID numérico.")
else:
    # Caso contrário, alerta sobre linhas sem código e incrementa o contador de erros
    print(f"❌ Erro Crítico: Existem {nulos_setor} linhas sem código de setor.")
    erros += 1

# 3. Teste dos Limites Matemáticos: nenhum valor pode ser negativo ou maior que 1
colunas_indices = [
    'ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado', 
    'ind_analfabetismo', 'ind_cor_raca', 'ind_densidade_habitacional', 
    'ind_pobreza_multidimensional'
# Lista de colunas dos índices a serem testados
]

# Conta quantos valores negativos existem nas colunas de índices
negativos = (df_teste[colunas_indices] < 0).sum().sum()
# Conta quantos valores acima de 1 existem nas colunas de índices
acima_um = (df_teste[colunas_indices] > 1).sum().sum()

if negativos == 0 and acima_um == 0:
    # Se todos os valores estão entre 0 e 1, imprime mensagem de sucesso
    print("✅ Matemática Imaculada: Todos os índices estão rigorosamente entre 0.0 e 1.0.")
else:
    # Caso contrário, alerta sobre valores fora dos limites e incrementa o contador de erros
    print(f"❌ Erro Matemático: Foram encontrados {negativos} valores negativos e {acima_um} valores > 1.")
    erros += 1

# 4. Teste da Morfologia: verifica se a coluna de moradia está presente
if 'Moradia_Predominante' in df_teste.columns:
    # Se a coluna existe, imprime mensagem de sucesso
    print("✅ Morfologia Urbana: A coluna de tipo de habitação está presente.")
else:
    # Caso contrário, alerta sobre ausência da coluna e incrementa o contador de erros
    print("❌ Erro Crítico: A coluna 'Moradia_Predominante' desapareceu.")
    erros += 1

# Imprime separador visual para o status final
print("-----------------------------------------")
if erros == 0:
    # Se não houve erros, imprime mensagem de aprovação final
    print("🏆 STATUS FINAL: EXCEL APROVADO! O PROJETO ESTÁ PRONTO PARA A REUNIÃO.")
else:
    # Caso contrário, alerta sobre inconsistências encontradas
    print(f"⚠️ ATENÇÃO: {erros} inconsistências detectadas. Reveja o código anterior.")


      AUDITORIA FINAL DA BASE EXCEL      
✅ Volume Perfeito: A base contém exatamente os 450088 setores válidos previstos.
✅ Integridade Chave: Nenhum setor censitário perdeu o seu ID numérico.
✅ Matemática Imaculada: Todos os índices estão rigorosamente entre 0.0 e 1.0.
✅ Morfologia Urbana: A coluna de tipo de habitação está presente.
-----------------------------------------
🏆 STATUS FINAL: EXCEL APROVADO! O PROJETO ESTÁ PRONTO PARA A REUNIÃO.
